# 09. 기하학적 변환

이동·전단·크기·회전·어파인·투시 변환과 문서 스캔 원리를 실습합니다.

강의 슬라이드의 코드를 실행 순서에 맞게 정리한 실습 노트북입니다.

> 이미지 예제는 노트북과 같은 위치에 `data` 폴더를 만들고 강의에서 사용하는 파일을 넣어 실행하세요.  
> `cv2.imshow()`와 카메라·마우스 예제는 데스크톱 Jupyter/VS Code 환경에서 실행하는 것을 권장합니다.


In [ ]:
from pathlib import Path
import math
import cv2
import numpy as np
import matplotlib.pyplot as plt

def read_image(name):
    image = cv2.imread(str(Path("data") / name))
    if image is None:
        raise FileNotFoundError(f"data/{name}")
    return image

def show(images, titles):
    fig, axes = plt.subplots(1, len(images), figsize=(6 * len(images), 5))
    axes = np.atleast_1d(axes)
    for ax, image, title in zip(axes, images, titles):
        ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        ax.set_title(title); ax.axis("off")
    plt.show()


## 이동과 전단


In [ ]:
src = read_image("tekapo.bmp")
h, w = src.shape[:2]

translation = np.array([[1, 0, 200], [0, 1, 100]], dtype=np.float32)
moved = cv2.warpAffine(src, translation, (w + 200, h + 100))

shear = np.array([[1, 0.5, 0], [0, 1, 0]], dtype=np.float32)
sheared = cv2.warpAffine(src, shear, (w + int(h * 0.5), h))
show([src, moved, sheared], ["source", "translation", "shear"])


## 크기 변경과 보간


In [ ]:
rose = read_image("rose.bmp")
nearest = cv2.resize(rose, None, fx=2, fy=2, interpolation=cv2.INTER_NEAREST)
linear = cv2.resize(rose, None, fx=2, fy=2, interpolation=cv2.INTER_LINEAR)
cubic = cv2.resize(rose, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
lanczos = cv2.resize(rose, None, fx=2, fy=2, interpolation=cv2.INTER_LANCZOS4)
show([nearest, linear, cubic, lanczos], ["nearest", "linear", "cubic", "lanczos"])


## 중심 기준 회전


In [ ]:
center = (w / 2, h / 2)
rotation = cv2.getRotationMatrix2D(center, 20, 0.7)
rotated = cv2.warpAffine(src, rotation, (w, h))
show([src, rotated], ["source", "rotate 20°, scale 0.7"])


## 어파인 변환


In [ ]:
src_points = np.float32([[0, 0], [w - 1, 0], [0, h - 1]])
dst_points = np.float32([[80, 80], [w - 120, 30], [40, h - 70]])
matrix = cv2.getAffineTransform(src_points, dst_points)
affine = cv2.warpAffine(src, matrix, (w, h))
show([src, affine], ["source", "affine"])


## 투시 변환으로 명함 펴기


In [ ]:
card = read_image("pinkwink_namecard.png")
out_w, out_h = 720, 400
src_quad = np.array([[360, 345], [879, 404], [895, 664], [254, 573]], np.float32)
dst_quad = np.array([[0, 0], [out_w - 1, 0],
                     [out_w - 1, out_h - 1], [0, out_h - 1]], np.float32)

perspective = cv2.getPerspectiveTransform(src_quad, dst_quad)
flattened = cv2.warpPerspective(card, perspective, (out_w, out_h))
show([card, flattened], ["source", "perspective corrected"])


## 문서 스캔 기본 함수

선택한 네 모서리를 A4 비율의 평면으로 변환합니다.


In [ ]:
def scan_document(image, corners, output_width=500):
    output_height = round(output_width * 297 / 210)
    source = np.asarray(corners, dtype=np.float32)
    target = np.array([
        [0, 0], [0, output_height - 1],
        [output_width - 1, output_height - 1], [output_width - 1, 0]
    ], dtype=np.float32)
    matrix = cv2.getPerspectiveTransform(source, target)
    return cv2.warpPerspective(image, matrix, (output_width, output_height),
                               flags=cv2.INTER_CUBIC)

document = read_image("scanned.jpg")
dh, dw = document.shape[:2]
corners = [[30, 30], [30, dh - 30], [dw - 30, dh - 30], [dw - 30, 30]]
scanned = scan_document(document, corners)
show([document, scanned], ["document", "scanned"])
